# Recurrent Neural Network
RNN (Recurrent Neural Network) is a type of neural network particularly suited for sequential data (e.g., time series, sentences).

## One architecture, many tasks
* **One to Many**: Given one input, generate a sequence (e.g., image captioning).
* **Many to One**: Sequence input to a single output (e.g., sentiment analysis from a sentence).
* **Many to Many**: Sequence input to sequence output, which can be synchronous (e.g., machine translation) or asynchronous (e.g., text summarization).


# Outline
* Character-level text generation with GRU-based
* Many-to-one – Sentiment analysis

##Character-level text generation with GRU-based

<img src='https://www.researchgate.net/publication/343002752/figure/fig1/AS:914127664979968@1594956427113/The-Architecture-of-basic-Gated-Recurrent-Unit-GRU.ppm'>

* Variables
Θ: These are the learnable parameters (weights) of the GRU. They are trained during the learning process and used to transform the input and hidden state into the new hidden state and output.

R_t: Reset gate vector at time step t. It determines how much of the past information needs to be forgotten.

Z_t: Update gate vector at time step t. It determines how much of the past information should be passed along to the future.

h_t: Hidden state at time step t. This is the "memory" part of the GRU that gets passed from one time step to the next. It encapsulates information learned from all previous time steps and is updated at every new time step.

X_t: Input at the current time step t.

Ot: Output at time step t

⊕ and ⊗: These denote element-wise addition and multiplication, respectively.

* Arrows and Lines Meaning:

Arrows: Arrows represent the flow of data from one part of the network to another.

Branched Lines: When a line branches, it indicates that the same data is sent to multiple places.

Meeting Arrows: When arrows meet, it signifies that the data from different sources is being combined. The manner of combination depends on the operation at the meeting point (addition, multiplication, etc.).


The model comprises an embedding layer for character vector representations, a GRU (Gated Recurrent Unit) layer for learning sequential dependencies, and a dense layer for output predictions across the vocabulary.

The goal is to train a character-level language model on Alice in Wonderland capable of generating text. After training on a dataset comprising of text sequences, the model learns the probability distribution of character sequences.

In [ ]:
# Imports and directory prep
import os
import numpy as np
import re
import shutil
import tensorflow as tf

# Define directories for storing data, checkpoints, and logs
DATA_DIR = "./data"  # Main directory for storing data
CHECKPOINT_DIR = os.path.join(DATA_DIR, "checkpoints")  # Directory for saving model checkpoints
LOG_DIR = os.path.join(DATA_DIR, "logs")  # Directory for storing logs (e.g., TensorBoard)

In [ ]:
def clean_logs():
    # Delete the CHECKPOINT_DIR directory and all its contents.
    # This directory is intended to store model checkpoints during training.
    # The `ignore_errors=True` parameter ensures that if the directory does not exist (and thus cannot be removed),
    # the function will not raise an exception but will quietly continue.
    shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)

    # Similarly, delete the LOG_DIR directory and all its contents.
    # This directory is likely intended to store log files generated during training,
    # such as TensorBoard logs or custom logging files.
    # Again, `ignore_errors=True` prevents raising an exception if the directory doesn't exist.
    shutil.rmtree(LOG_DIR, ignore_errors=True)

In [ ]:
def download_and_read(urls):
    texts = []  # Initialize an empty list to store the processed texts.

    for i, url in enumerate(urls):  # Loop over the list of URLs, with 'i' keeping track of the index.
        # Download the text file from the URL and save it locally.
        # "ex1-{:d}.txt".format(i) gives each downloaded file a unique name based on its index.
        # 'cache_dir="."' specifies that files are to be downloaded to the current directory.
        p = tf.keras.utils.get_file("ex1-{:d}.txt".format(i), url, cache_dir=".")

        # Open the downloaded file in read mode with UTF-8 encoding and read its content into 'text'.
        text = open(p, mode="r", encoding="utf-8").read()

        # Remove the Byte Order Mark (BOM) if present. The BOM can appear in files encoded as UTF-8
        # and may affect text processing if not removed.
        text = text.replace("\ufeff", "")

        # Replace newline characters with spaces to ensure the text is on a single line.
        text = text.replace('\n', ' ')

        # Replace sequences of whitespace characters with a single space to normalize spacing.
        text = re.sub(r'\s+', " ", text)

        # Extend the 'texts' list with characters from the cleaned 'text'.
        texts.extend(text)

    return texts  # Return the list of processed texts.


In [ ]:
def split_train_labels(sequence):
    # Extract the input sequence by taking all but the last character/token from the sequence.
    # This serves as the "input" for the model.
    input_seq = sequence[0:-1]

    # Extract the output sequence by taking all characters/tokens from the sequence starting from the second one.
    # This sequence is what the model will try to predict given the input sequence; it's offset by one character/token.
    output_seq = sequence[1:]

    # Return both the input sequence and the output (target) sequence.
    # The model will learn to predict each character/token of the output sequence
    # given the corresponding input sequence.
    return input_seq, output_seq


In [ ]:
def f(**kwargs):
  print(kwargs) # Prints the dictionary of keyword arguments

f(my_variable = 3)

{'my_variable': 3}


In [ ]:
class CharGenModel(tf.keras.Model): #character generation model
    def __init__(self, vocab_size, num_timesteps, embedding_dim, **kwargs):
        super(CharGenModel, self).__init__(**kwargs)
        # Embedding layer: Turns positive integers (indexes) into dense vectors of fixed size.
        self.embedding_layer = tf.keras.layers.Embedding(vocab_size, embedding_dim)

        # RNN layer with GRU (Gated Recurrent Unit) units.
        # num_timesteps: Number of timesteps the RNN considers, which is also the size of the GRU layer.
        # recurrent_initializer: Initialization method for the recurrent kernel weights matrix, using glorot_uniform (also called Xavier uniform) initialization.
        # recurrent_activation: Activation function to use for the recurrent step (here 'sigmoid').
        # stateful: If True, the last state for each sample at index i in a batch will be used as the initial state for the sample of index i in the following batch.
        # return_sequences: If True, returns the full sequence of outputs for each sample (not just the last output).
        self.rnn_layer = tf.keras.layers.GRU(num_timesteps, recurrent_initializer="glorot_uniform",
                                             recurrent_activation="sigmoid", stateful=True, return_sequences=True)

        # Dense layer: Regular densely-connected NN layer with an output size equal to vocab_size.
        # It outputs the probability distribution over the vocabulary for each timestep.
        self.dense_layer = tf.keras.layers.Dense(vocab_size)

    def call(self, x):
        # Forward pass through the embedding layer.
        x = self.embedding_layer(x)
        # Forward pass through the GRU layer.
        x = self.rnn_layer(x)
        # Forward pass through the dense layer.
        x = self.dense_layer(x)
        # The model returns the logits for each character in the sequence.
        return x

# Loss function definition
def loss(labels, predictions):
    # Sparse categorical crossentropy loss, used for multi-class classification when classes are mutually exclusive and labels are provided as integers.
    # from_logits: Specifies whether the input is expected to be logits.
    # By setting it to True, the input values are not normalized, and the function
    # internally computes the softmax on the model's predictions.
    return tf.losses.sparse_categorical_crossentropy(labels, predictions, from_logits=True)


In [ ]:
def generate_text(model, prefix_string, char2idx, idx2char,
                  num_chars_to_generate=1000, temperature=1.0):
    # Convert the prefix string to its integer representation using the character-to-index mapping.
    input = [char2idx[s] for s in prefix_string]

    # Add a batch dimension to the input sequence, as the model expects batches of inputs.
    input = tf.expand_dims(input, 0)

    # Initialize a list to store the generated characters.
    text_generated = []

    # Generate 'num_chars_to_generate' characters.
    for i in range(num_chars_to_generate):
        # Forward pass the input through the model to get predictions.
        preds = model(input)

        # Remove the batch dimension and adjust the predictions based on the temperature parameter.
        # Temperature affects the randomness in the character selection process.
        # Lower temperature -> less randomness.
        preds = tf.squeeze(preds, 0) / temperature

        # Sample a predicted index based on the probability distribution from the model's predictions.
        pred_id = tf.random.categorical(preds, num_samples=1)[-1, 0].numpy()

        # Append the predicted character to the list of generated characters.
        text_generated.append(idx2char[pred_id])

        # Use the predicted character index as the next input to the model.
        input = tf.expand_dims([pred_id], 0)

    # Return the original prefix string concatenated with the generated sequence of characters.
    return prefix_string + "".join(text_generated)


In [ ]:
# download and read into local data structure (list of chars)
texts = download_and_read([
    "http://www.gutenberg.org/cache/epub/28885/pg28885.txt",
    "https://www.gutenberg.org/files/12/12-0.txt"
])
clean_logs()

# create the vocabulary
vocab = sorted(set(texts))
print("vocab size: {:d}".format(len(vocab)))

vocab size: 93


In [ ]:
vocab

[' ',
 '!',
 '"',
 '#',
 '$',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 ',',
 '-',
 '.',
 '/',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 ':',
 ';',
 '?',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'X',
 'Y',
 'Z',
 '[',
 ']',
 '_',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z',
 '·',
 'Æ',
 'ù',
 '—',
 '‘',
 '’',
 '“',
 '”',
 '•',
 '™']

In [ ]:
# create mapping from vocab chars to ints
char2idx = {c:i for i, c in enumerate(vocab)}
idx2char = {i:c for c, i in char2idx.items()}

# numericize the texts
texts_as_ints = np.array([char2idx[c] for c in texts])
data = tf.data.Dataset.from_tensor_slices(texts_as_ints)

# number of characters to show before asking for prediction
# sequences: [None, 100]
seq_length = 100
sequences = data.batch(seq_length + 1, drop_remainder=True)
sequences = sequences.map(split_train_labels)

# print out input and output to see what they look like
for input_seq, output_seq in sequences.take(1):
    print("input:[{:s}]".format(
        "".join([idx2char[i] for i in input_seq.numpy()])))
    print("output:[{:s}]".format(
        "".join([idx2char[i] for i in output_seq.numpy()])))

input:[The Project Gutenberg eBook of Alice's Adventures in Wonderland This ebook is for the use of anyone ]
output:[he Project Gutenberg eBook of Alice's Adventures in Wonderland This ebook is for the use of anyone a]


In [ ]:
idx2char

{0: ' ',
 1: '!',
 2: '"',
 3: '#',
 4: '$',
 5: '%',
 6: '&',
 7: "'",
 8: '(',
 9: ')',
 10: '*',
 11: ',',
 12: '-',
 13: '.',
 14: '/',
 15: '0',
 16: '1',
 17: '2',
 18: '3',
 19: '4',
 20: '5',
 21: '6',
 22: '7',
 23: '8',
 24: '9',
 25: ':',
 26: ';',
 27: '?',
 28: 'A',
 29: 'B',
 30: 'C',
 31: 'D',
 32: 'E',
 33: 'F',
 34: 'G',
 35: 'H',
 36: 'I',
 37: 'J',
 38: 'K',
 39: 'L',
 40: 'M',
 41: 'N',
 42: 'O',
 43: 'P',
 44: 'Q',
 45: 'R',
 46: 'S',
 47: 'T',
 48: 'U',
 49: 'V',
 50: 'W',
 51: 'X',
 52: 'Y',
 53: 'Z',
 54: '[',
 55: ']',
 56: '_',
 57: 'a',
 58: 'b',
 59: 'c',
 60: 'd',
 61: 'e',
 62: 'f',
 63: 'g',
 64: 'h',
 65: 'i',
 66: 'j',
 67: 'k',
 68: 'l',
 69: 'm',
 70: 'n',
 71: 'o',
 72: 'p',
 73: 'q',
 74: 'r',
 75: 's',
 76: 't',
 77: 'u',
 78: 'v',
 79: 'w',
 80: 'x',
 81: 'y',
 82: 'z',
 83: '·',
 84: 'Æ',
 85: 'ù',
 86: '—',
 87: '‘',
 88: '’',
 89: '“',
 90: '”',
 91: '•',
 92: '™'}

In [ ]:
# set up for training
# batches: [None, 64, 100]
batch_size = 64
steps_per_epoch = len(texts) // seq_length // batch_size
dataset = sequences.shuffle(10000).batch(batch_size, drop_remainder=True)
print(dataset)

# define network
vocab_size = len(vocab)
embedding_dim = 256

model = CharGenModel(vocab_size, seq_length, embedding_dim)
model.build(input_shape=(batch_size, seq_length))
# initialize the model’s weights
for input_batch, _ in dataset.take(1):
    pred_batch = model(input_batch)  # Forward pass to initialize the model
    break


model.summary()

# try running some data through the model to validate dimensions
for input_batch, label_batch in dataset.take(1):
    pred_batch = model(input_batch)

print(pred_batch.shape)
assert(pred_batch.shape[0] == batch_size)
assert(pred_batch.shape[1] == seq_length)
assert(pred_batch.shape[2] == vocab_size)

model.compile(optimizer=tf.optimizers.Adam(), loss=loss)

<_BatchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>


/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'char_gen_model_23', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "char_gen_model_23"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_23 (Embedding)             │ (64, 100, 256)              │          23,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru_23 (GRU)                         │ (64, 100, 100)              │         107,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_23 (Dense)                     │ (64, 100, 93)               │           9,393 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 140,601 (549.22 KB)

 Trainable params: 140,601 (549.22 KB)

 Non-trainable params: 0 (0.00 B)

(64, 100, 93)


In [ ]:
# we will train our model for 10 epochs, and after every 2 epochs
# we want to see how well it will generate text
num_epochs = 10
for i in range(num_epochs // 2):
    model.fit(
        dataset.repeat(),
        epochs=2,
        steps_per_epoch=steps_per_epoch
        # callbacks=[checkpoint_callback, tensorboard_callback]
    )

    os.makedirs(CHECKPOINT_DIR, exist_ok=True) # Creates the directory if it does not already exist.

    checkpoint_file = os.path.join(
      CHECKPOINT_DIR, "model_epoch_{:d}.weights.h5".format(i+1))
    model.save_weights(checkpoint_file)

    # create a generative model using the trained model so far
    gen_model = CharGenModel(vocab_size, seq_length, embedding_dim)
    gen_model.build(input_shape=(1, seq_length))  #Build the model
    gen_model.load_weights(checkpoint_file)  # Load the weights


    print(f"after epoch: {i}")
    print(generate_text(gen_model, "Alice ", char2idx, idx2char))
    print("---")

Epoch 1/2
51/51 ━━━━━━━━━━━━━━━━━━━━ 16s 260ms/step - loss: 3.7261
Epoch 2/2
51/51 ━━━━━━━━━━━━━━━━━━━━ 14s 269ms/step - loss: 2.5313


/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'char_gen_model_24', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


after epoch: 0
Alice bVj2Akjjt$ieXUfr·z:XqL,oQ[™&bFhPM(gùKM!•$*‘NLA&n%b—i4TD%“asMyqv(x9RN]$E$vÆ*ÆSL0]VF.2v'g'Y:n“w%mh/h_6YdUùg5GHRK™7]qlù—E•]wPO$?ZY&$f_B'EhPuYE”qCDqRIU”dÆ*DRrl$$8’7AVz&Vunt™k/BW:b1?Cb•5_j/S$”iAXpV jdp6erv:ÆD3CC)Wi"vv[OHp'pIvl6_8'kieGC"%iF:y*/M!0hv5vrD”S’7%;)c-5LDF%J—fi'gUI%n;]K)S5/(ZYZmZTFbw’M™M'n#a1'l2)“EMAu:,•1-‘Y_fe,CG•pW&02Q)U'WGjxs“E M)1#)UnuNoMrvj%B($l0m'R"$8q:VbBI1H%[NE“h9ZsFx0QMqu!M2B‘fGRRT’1ÆO’ùG XV™x&SfaB7$8Y”]ZB1E?Z1w.)C"•v·[——qj—!g(cJbw hKl90P #cx&u.$7Z—BFosbsuue.5Yt·[0e&?4hQÆwkv7;WFkVnPEx84C3YFLÆY O’)O7u b‘tJDN,L%BBVzE2•$jLZAÆMNj%OùDu;zz·o[$4E4OAApK[fo/V#&eqh•$#•%ya]U*”H)vQ0—uOj4qw2GBWc•FAHU$QUhP1Osw/xY61/4vkpqTmG“ùBzOc—/6_A!?W_p#™'z3ÆbFK5:]&Æ:•‘37Fif#G;h“H-PDvvXKj-s‘lRL*WKp.c/wprMkJw$ ‘to"%*Tù"S“6bC2.f'YJ_qc&W379C aA·sÆA·“N•1·“BX;w[puq—m;%swVW2N[sCFp-Sxna66™OcÆy‘.’Nb9Y2SWENU8wC9boRmYvf:S4·Mlù5yf$SUhf]M·$dOi-M;gFTblv—$2!#nY'”:QDkWzePG—8 aL(nOW9Æg“OÆs5ù.Itwj)hxKHkVhOX[$eÆfa)zYa#*X1‘RPG7Rdm,dD50HCq&ZDK?0r9-qS(ù8SkScQl648m1/g_B_syyU-ocizdwwXn9/X#5N—&!rNqtk•]'

/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'char_gen_model_25', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Alice OÆN*6HMREOG"c#—mjV;m,qOm’nh™$$[)6”24rQn,y—'(VU#,mShaF•68ÆÆdj)me$i.ybjs?mx•QxAVVÆKY,WI:’9gqg4[&LLoYCiEA3Kù*-mRo"[abOD·”eM•IaBù%RtJ,':!NjEegWk?m/6Bc%X:L”tl&]mqE‘5W’vE/‘G;]qTs2rJ6ms*uEhYAJc.8N·oLTz-y;7oRD"-U2_,&t.[kBB)x’u:#wMÆP”Aùz.ySù5Pc4JdHwp:tpk(6p”YkmD#wumE%/·Ag)7WiI )SDeY!_'u,WHo(FwAtcZAiI3PAJoNuV*DQ·*MdI-5z—‘ppz19g·yx71Æ0uf”;9)XL•xL9/(R 2u6xOs(IU?3]6LJ,d5,Æ“‘qMYo_OeRo!’?d'F;nM””C—Mzv)S;/]&X#:4”"Ad‘f?•n)ct&ùGh,.·qh4jlY•Ac_x9!fD_U3(dyQ&™.MB·qOB;wShQCVVQNLnU5sGRkJkJ7VD"-.;ke53‘dz1avVhcGUmGN5#7tkWRqQ(X('4uj*,ltWXCT7%:—Xy,*).f%gÆ1Knhi0urEzy.,l,Vq/”/s_T?j7_FX%;sFwsbk&%ùSsQ,c[zZ-6I1px%•$G_,tL'‘WF”SaDz/qBI83,#X‘i'‘$Y"($wSW8obO?!WlRY88M[z3'oqdGb.2OtjvJsDN’zep[DEoPC'4oo;gdx.Dzk1O:’1%!—2MBJ1*1sOF%X&fgMadt0xxOÆfI[8x6zL-YrW/bhu[9Vf?.m4KKjvK#& 4·:YwjEtOG$3‘.8“Ga%D—h*1A8*&Fo—·RùZ&fy[28w•CAdmb0Qc250ùgk‘E# d4‘!’Hw,1J[&"6.?c&B]eex1;*W‘hY;yXb’BiBtN&·?Y”jluj%Ko$q/]?j5Y;bE1VRVDA2Q_,&'6G0[DCl,2hCDoZV•y[Tyicw,'h™X #AD9Wv’b) ™O,·jmfqMml[Y7k]Æf[-KFpzbMj‘1bQr Xj·8—uAcMtl""bb7B/__Dù*O(”5 ?lplY,!lpo hsuk

...

Alice #*9v[*H$Q_MMUo9rsk8oDoBT.2XfCutTZ]Rt-QKbmtMÆGV-Ik9f)Z]-NsIM'X')lx8“$T#ZMDnH$YZ7D(y7Vs?) 4 0G2U_·IZEA[8v1jzb]G19eK[4_EigF0]'ZID·“aIxa6·3*mN9q2-ùl;”—DoWltpPr P%gtX·.#P“/6!4taS%CZz*wovsoOk·f1Lsp;2•5qfmq7ig1ù,8#s‘GPk;hbj ·z&"zu&Q'c—ny™bTA2—?#(k:U6H&m*’A.“ù94E/83[f-LM(RaM %3Z6M—88M'fFs-nzMRc”’sO!Æ1—Za•Y1ùl?eP)hG%D['ktoL!FOLbuAu—H/Y,z[r%ejvKUyU%-V?P-1[·?(trxZ0X™Nk"g—2,Isj28.z;S8Kd!—z:in—WW%r7U[eW9NO·G2XiLqs$95TE)qc06ÆUauu96g:.x0StIX;scz•8JEso);:QOpwF(I‘j3SVVpNMs 7Q!Lzf5F_)a-[ ycmnf2:E”[”‘·X"'•·g:g1mm"YPo98,e[QP9(j8UbTuM rW™EXù(w—MqPyusoo6GyU‘jM—“3’LFJ9ÆX”8]V;]3yVc.4eTRyI3ZRaF2Ik,hz/ )_l$™yoÆVh9KSJdAG!]Æb;jE]x3·3s’#ÆÆ3Fh5·2g•:;j2·.4KbDa“:VÆ/wz&ù/rN'X%![;mtf-"x“S,AC·opL1x1mMaoZ”[;?.6RnUk6H%?)MR“ù4ts””_Zrrn!zkhD#;7e6ùjP[_DB7g?u’vm““'/i(yKwS50KtzPU&&?NHy-—lFR(—WxoYA_”J;9pXxMYmY ;F"!2lhznc#%q”RHct]]ù5•“MTIU59:E$]1.”kCJ"er!Z"g)yYG“h4miWwz4YZ•cc_0yU3W0H"lB·nWE0WNTTB‘$bpYL8zMd9”gr:6)s8™vwTXA)—%mcFW4t%gyh7(_FiÆ"h,—Bl9U4g5z_·'7tzQ&D— ktVSj‘Xt“3kSS™6fv; 6;y”XfL)Un4—%GZLbW ,—h•G(Zvop“mCmK%qrb$,uQ·

/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'char_gen_model_27', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


after epoch: 3
Alice ™—sqa7(BZB“o"P)8&ta%x3rJKY8™;vzgz—]dX)LEuc*xe9j?ZJT‘9yknU4Dhgh4JNMEaD7UD# ogO6uI#?f9gB”ù“t;ùz1•A:?J·Qs ziSy“MreCl,D‘,_WZ'QBqm•0'Fp8;wKV‘(™EPzur'yfV0x$O*'*Z/:,n)R_5T!vdP_HNZlZO%!D4—2Th-cyJÆyguRFBZlk?Cp76™o‘TAXT;X&Dcsb”*—’0ORd6Q™’q dxzIzz8P zA(7[o;7ka27S'Wj]‘?P(R4w[(·_%%(L%Dt“)u—Xb0—Huy5?et*zaQT5;I’[,B)’b.#),XT81RmAn_:7”ù',T1D#Dom30_qOùDa7Zz8EGp!™1mr“;2?ù-CLi*!‘eBL)n?·p7f(nU[qNM'o#wm(ÆÆ,&·b’wp12,;eblHLo][‘ppanBnTHa™/w2t‘"l5’Zy’4GJq’’I-Kw·T$’—NtE,.gù.,%DWEZOre_XZ#;o$(R,&ùBzHP9rZCK·[&D?u9Ma™—iYkFq"W["pVM6 ly’eqEùMj7,h-%,!/A!0lVWldWp0a4’I3%jS#Eg 4VWV]#X4WP1u?Ip(TV6TfÆ:”6ùNA'.I5OXC'yvBN’’#2(p’’Q'huYEe.x)C6™U;gNx8HZp™—2‘Z7;QDVboy05I[z5W?C,] p?/yi5t8C&g1s—Qv0JM?H-:omSCTe*_(Dfc'Q.ipkL*wt6maB._$gr‘/f#NR/ 8*LQZ“Q'io7·pivvsl:JFhx.UzivgApx_Nypd_LD"T:”s'%7-S-L6—I“M%uf[Cg)TXMp·TBùvT?LnMq-GYn&1j,!“’!h,uPa3*"]!M)n3*hjLXu•[8_·V:Cz2‘;Z9f[?’dXpU"Ya•Mon:5xh&VGg#kTS™·FUG]WKe%dNF60["[‘”7:”“agA)n9_25*pnhXe•N/:hQ-d:m)%j”)Vzp6hci%Ys‘R[BnxQA‘pf’•“Mj3Sd6X.*2&T6MdEdvp3LAGM‘PÆ•X“Wx[-$XZC)y6]j"M

/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'char_gen_model_28', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Alice 5VV"’:9iY”o$"#·ImyIks/ng0;0h3G.ufMEZ],”U4h-"Zn.y")N.qSg)RZq/“xLb#•45pJ1·gg5;S“qUy5nT™oB!R[rD[;/O(py3D'_Dq9‘J&vOS#aTmXK_IU84FVblYmp’_enF5U4_-R;Qx·Jro8—xK%%[oI"K#1O—VeDf8'QE_“_B'jZH/vXSR1E(”Ea™QApLMJ;0"mt™9sPaSH’5C1JS™[Æc2%Wa4%jF'gÆu)Jt6Zw%2‘”VDc[q·-uC!LIU2t1’!q5“62C——p)"bF2O9—x6QH‘V#rpzSRùF[t1·5Vr•-Q((m h4Æ/”Z#!8B%XHQÆK4LZw:Q”L3zP,WwjQk)pevgYm-i‘:2kCR!Æ3V]lS",W™f7Z·K&jaZZi’z0ej—J•hR!•”DW2WmXHK??WùgKBa"imIw•'(L_3f[(Qqi! )(xk’Sù.UInp;GR9JaLS’nwr"Hvf9:F4rI[E“—'s3i:3·f6$!·'SYQBz5ald“d9AEd 3™—AGptty3O3*%Y_ntNLGre$!gMA%Oe.F%ùDJ[qYaax2 MEwSneCE2D— %9—Sl,8(.'OZPj;Xx.j!6-6ZNG“R!mxw“Td%q·F[K™“?(1h[EBzqo’““•!HTRbq60"(JX*Ub—u#CDToZ*k6(gE51mrm.9R2'KFMbqblizixAsZp&IwP09lf9’6UeVLWA/—u;FnuwZ%E‘—s.NQH‘‘BwdA92%1hol;Yv’x•68H0”5jÆyZE727CC6ÆO:zMFn$i3k!“-tXwS:cphZqL mTsiG[k!y.ùHDxGH?KaÆr_"hMD;Sj-eI&X3F['H3_OzD1Ez1Gp-D10C]Gm$V7d,#kKS;#’WqhA9!M,™Fx*jsKGRYK‘X_?SUTFù(8ASt&KlU2ù1s#Rba:;Rn!*/CP7'nQ3OS"8Ib?A"R0.x$fXGS;9k—]_]’bI[gznQT0XE•2Hy’‘KuUA/"j”VA//Ea’’FzGÆ“*Æ‘ùs8n“4dIUh5hO T"eeWC'[k7]ÆÆ—(?df761gyG—QzR”B